# Data Preparation

In [256]:
# import libraries
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns

## 1. Data Understanding, Cleaning & Transformation

In [257]:
# Ingest data
fuel_mix_elec_gen = pd.read_csv('C:/Users/Admin/Desktop/net-zero emission by 2025 project/net-zero-emission-data-analytics-project/data sources/data.gov.sg/AnnualFuelMixforElectricityGenerationbyEnergyProducts2005toJun2021.csv')
elec_gen_and_consump = pd.read_csv('C:/Users/Admin/Desktop/net-zero emission by 2025 project/net-zero-emission-data-analytics-project/data sources/data.gov.sg/ElectricityGenerationAndConsumptionAnnual.csv')
ggas_emission_gas_type = pd.read_csv('C:/Users/Admin/Desktop/net-zero emission by 2025 project/net-zero-emission-data-analytics-project/data sources/data.gov.sg/GreenhouseGasEmissionsByGasTypeAnnual.csv')
ggas_emission_sector = pd.read_csv('C:/Users/Admin/Desktop/net-zero emission by 2025 project/net-zero-emission-data-analytics-project/data sources/data.gov.sg/GreenhouseGasEmissionsBySectorAnnual.csv')
per_capita_gdp_chained = pd.read_csv('C:/Users/Admin/Desktop/net-zero emission by 2025 project/net-zero-emission-data-analytics-project/data sources/data.gov.sg/PerCapitaGDPInChained2015DollarsAnnual.csv')
renewable_energy_share = pd.read_csv('C:/Users/Admin/Desktop/net-zero emission by 2025 project/net-zero-emission-data-analytics-project/data sources/data.gov.sg/RenewableEnergyShareInTheTotalFinalEnergyConsumptionAnnual.csv')

In [258]:
# See number of records and columns in each dataset
datasets = {
    "Annual Fuel Mix for Electricity Generation by Energy Products 2005 to Jun 2021": fuel_mix_elec_gen,
    "Electricity Generation and Consumption Annual": elec_gen_and_consump,
    "Greenhouse Gas Emissions by Gas Type Annual": ggas_emission_gas_type,
    "Greenhouse Gas Emissions by Sector Annual": ggas_emission_sector,
    "Per Capita GDP in Chained 2015 Dollars Annual": per_capita_gdp_chained,
    "Renewable Energy Share in Total Final Energy Consumption Annual": renewable_energy_share,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

Annual Fuel Mix for Electricity Generation by Energy Products 2005 to Jun 2021: 68 rows, 3 columns
Electricity Generation and Consumption Annual: 18 rows, 52 columns
Greenhouse Gas Emissions by Gas Type Annual: 15 rows, 25 columns
Greenhouse Gas Emissions by Sector Annual: 13 rows, 5 columns
Per Capita GDP in Chained 2015 Dollars Annual: 8 rows, 67 columns
Renewable Energy Share in Total Final Energy Consumption Annual: 1 rows, 16 columns


From the above output, we can see that datasets are small, with per capita GDP dataset having the most records of 67. Next, we will look at each dataset to understand more about their data.

In [259]:
# Look at Annual Fuel Mix for Electricity Generation by Energy Products 2005 to Jun 2021 dataset
fuel_mix_elec_gen.head(5)

,year,energy_products,percentage
0,2005,Petroleum Products,23.1
1,2005,Natural Gas,74.4
2,2005,Coal,0.0
3,2005,Others,2.5
4,2006,Petroleum Products,19.8


The output above shows that the dataset has *year*, *energy_products*, and *percentage* columns, showing contribution of electricity generation in % each year for each energy product. We will not transform this dataset as the data is already easily understandable and with proper columns. 

Before moving on to look at next dataset, we will **check whether there are any missing or duplicate records** in this dataset.

In [260]:
# Check missing values
fuel_mix_elec_gen.isna().sum()


year               0
energy_products    0
percentage         0
dtype: int64

In [261]:
# Check duplicate values
fuel_mix_elec_gen.duplicated().sum()

0

From the above cells, we can see that there are no missing or duplicate records present in the dataset. We will also make sure data types are correct to ensure high data quality.

In [262]:
# Check data types
fuel_mix_elec_gen.dtypes

year                 int64
energy_products     object
percentage         float64
dtype: object

The output above shows the correct data types. Lastly, for this dataset, we will make sure text column (*energy_products*) is clean and make sure no truncation is needed by checking any leading or trailing whitespaces in the values. 

In [263]:
# Check for leading and trailing spaces in energy_products column
fuel_mix_elec_gen['energy_products'].unique()

array(['Petroleum Products', 'Natural Gas', 'Coal', 'Others'],
      dtype=object)

We can see from the above output that the values have no leading or trailing whitespaces, therefore, there is no need for truncation. Next, let's take a look at Electricity Generation and Consumption Annual dataset.

In [264]:
# Look at Electricity Generation and Consumption Annual dataset
elec_gen_and_consump.head()

,DataSeries,2025,2024,2023,2022,2021,2020,2019,2018,2017,...,1984,1983,1982,1981,1980,1979,1978,1977,1976,1975
0,Electricity Generation,60410.4,59616.1,57384.2,57113.7,55788.3,53071.6,54142.3,52904.8,52225.8,...,9420.7,8625.9,7859.5,7441.9,6940.5,6447.8,5897.9,5114.7,4604.9,4175.7
1,Electricity Consumption,na,57639.7,55426.4,54910.6,53495.6,50789.7,51736.7,50466.9,49643.7,...,na,na,na,na,na,na,na,na,na,na
2,Industrial-Related,na,22704.0,22103.0,22693.9,22293.1,20978.9,21463.9,21441.8,21240.6,...,na,na,na,na,na,na,na,na,na,na
3,Manufacturing,na,20630.5,20171.2,20682.1,20284.8,19121.3,19439.4,19441.0,19306.8,...,na,na,na,na,na,na,na,na,na,na
4,Construction,na,440.0,421.6,464.5,472.0,387.0,397.1,431.5,485.5,...,na,na,na,na,na,na,na,na,na,na


As we can see from the output, this dataset has DataSeries column, which shows electricity consumption by sectors, along with total electricity generation and consumption value, in Gigawatt Hours, with the rest of the columns being year columns from 1975 to 2025. Therefore, this needs to be transformed, into one year column.

We will also add a category column with the value 'Total' (for Electricity Generation and Electricity Consumption rows in DataSeries column) or 'Consumption Sector' (for the rest of the rows in DataSeries column). Therefore, the final transformed dataset will have:
1. Year
2. Category
3. DataSeries
4. Value (in Gigawatt Hours)

Before, transforming, we will quickly check for **missing or duplicate values** in the dataset. If there are none, we will proceed to transform the dataset as discussed.

In [265]:
# Check missing values
elec_gen_and_consump.isna().sum()

DataSeries    0
2025          0
2024          0
2023          0
2022          0
2021          0
2020          0
2019          0
2018          0
2017          0
2016          0
2015          0
2014          0
2013          0
2012          0
2011          0
2010          0
2009          0
2008          0
2007          0
2006          0
2005          0
2004          0
2003          0
2002          0
2001          0
2000          0
1999          0
1998          0
1997          0
1996          0
1995          0
1994          0
1993          0
1992          0
1991          0
1990          0
1989          0
1988          0
1987          0
1986          0
1985          0
1984          0
1983          0
1982          0
1981          0
1980          0
1979          0
1978          0
1977          0
1976          0
1975          0
dtype: int64

In [266]:
# Check duplicate values
elec_gen_and_consump.duplicated().sum()

0

Because there are no missing or duplicate records, we can now proceed to do the transformation.

In [267]:
# Transform Electricity Generation and Consumption Annual dataset
year_cols = [col for col in elec_gen_and_consump.columns if col != 'DataSeries']

elec_gen_and_consump_transformed = elec_gen_and_consump.melt(
    id_vars=['DataSeries'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Value'
)

elec_gen_and_consump_transformed['Category'] = np.where(
    elec_gen_and_consump_transformed['DataSeries'].isin([
        'Electricity Generation',
        'Electricity Consumption'
    ]),
    'Total',
    'Consumption Sector'
)

# Convert Year to integer
elec_gen_and_consump_transformed['Year'] = (
    elec_gen_and_consump_transformed['Year']
    .astype(int)
)

# Convert Value to numeric and handle "na"
elec_gen_and_consump_transformed['Value'] = pd.to_numeric(
    elec_gen_and_consump_transformed['Value'],
    errors='coerce'
)

# Reorder columns
elec_gen_and_consump_transformed = elec_gen_and_consump_transformed[
    ['Year', 'Category', 'DataSeries', 'Value']
]

elec_gen_and_consump_transformed.head()

,Year,Category,DataSeries,Value
0,2025,Total,Electricity Generation,60410.4
1,2025,Total,Electricity Consumption,NaN
2,2025,Consumption Sector,Industrial-Related,NaN
3,2025,Consumption Sector,Manufacturing,NaN
4,2025,Consumption Sector,Construction,NaN


As we can see above, the transformation looks correct, giving *Year*, *Category*, *DataSeries*, and *Value* columns as we expected. We will also check data types of each column to maintain data quality for later analysis.

In [268]:
# Check data types
elec_gen_and_consump_transformed.dtypes

Year            int32
Category       object
DataSeries     object
Value         float64
dtype: object

We found out that data types are correct. Next, we will ensure there are no missing or duplicate values after transformation. First, we will check missing values.

In [269]:
# Check missing values
elec_gen_and_consump_transformed.isna().sum()

Year            0
Category        0
DataSeries      0
Value         527
dtype: int64

Interestingly, there are now missing values in *Value* column. We will find out why by checking the records that has missing values in *Value* column.

In [270]:
# Check which records has missing values in the column "Value"
elec_gen_and_consump_transformed[
    elec_gen_and_consump_transformed['Value'].isna()
]

,Year,Category,DataSeries,Value
1,2025,Total,Electricity Consumption,NaN
2,2025,Consumption Sector,Industrial-Related,NaN
3,2025,Consumption Sector,Manufacturing,NaN
4,2025,Consumption Sector,Construction,NaN
5,2025,Consumption Sector,Utilities,NaN
...,...,...,...,...
913,1975,Consumption Sector,"Professional, Scientific & Technical, ...",NaN
914,1975,Consumption Sector,Other Commerce And Service-Related,NaN
915,1975,Consumption Sector,Transport-Related,NaN
916,1975,Consumption Sector,Households,NaN


The output tells us that some years, such as 1975 and 2025 as we can see have no electricity consumption data. This aligns with the original data having 'na' values in them, except for Electricity Generation. We will confirm our doubt in the next cell by checking which year has how much missing values in the *Value* column.

In [271]:
# Check how many records has missing values in the column "Value" by Year
elec_gen_and_consump_transformed[
    elec_gen_and_consump_transformed['Value'].isna()
].groupby('Year').size()

Year
1975    17
1976    17
1977    17
1978    17
1979    17
1980    17
1981    17
1982    17
1983    17
1984    17
1985    17
1986    17
1987    17
1988    17
1989    17
1990    17
1991    17
1992    17
1993    17
1994    17
1995    17
1996    17
1997    17
1998    17
1999    17
2000    17
2001    17
2002    17
2003    17
2004    17
2025    17
dtype: int64

In [272]:
# Check number of unique values in the column "DataSeries"
elec_gen_and_consump_transformed['DataSeries'].nunique()

18

The results show that for the years **1975 to 2004** and **2025**, there are 17 missing values for each year. Since the DataSeries column contains 18 unique categories, this indicates that only Electricity Generation has recorded values for those years, while the remaining 17 data series contain missing values (NaN). 

This pattern suggests that electricity generation data is available for these years, whereas electricity consumption and its sectoral breakdown were not reported in the source dataset. Also, since imputing those values would make the data highly inaccurate, we will decide to remove those records with missing values.

In [273]:
# Remove missing values in the column "Value" since it is not possible to impute the missing values with a reasonable assumption.
elec_gen_and_consump_transformed = elec_gen_and_consump_transformed.dropna(
    subset=['Value']
)

We will now check the missing values again.

In [274]:
# Check missing values
elec_gen_and_consump_transformed.isna().sum()

Year          0
Category      0
DataSeries    0
Value         0
dtype: int64

The output above confirms that there are no more missing values in our dataset. Lastly, we will check *Category* and *DataSeries* columns to ensure there are no leading or trailing whitespaces.

In [275]:
# Check leading or trailing whitespaces for the column "Category"
elec_gen_and_consump_transformed['Category'].unique()

array(['Total', 'Consumption Sector'], dtype=object)

In [276]:
# Check leading or trailing whitespaces for the column "DataSeries"
elec_gen_and_consump_transformed['DataSeries'].unique()

array(['Electricity Generation', 'Electricity Consumption',
       '    Industrial-Related', '        Manufacturing',
       '        Construction', '        Utilities',
       '        Other Industrial-Related',
       '    Commerce And Service-Related',
       '        Wholesale And Retail Trade',
       '        Accommodation And Food Services',
       '        Information And Communications',
       '        Financial And Insurance Activities',
       '        Real Estate Activities',
       '        Professional, Scientific & Technical, Administration & Support Activities',
       '        Other Commerce And Service-Related',
       '    Transport-Related', '    Households', '    Others'],
      dtype=object)

The outputs show that there are leading whitespaces present in the *DataSeries* column. Therefore, we will truncate it.

In [277]:
# Truncate the column "DataSeries" to remove leading and trailing whitespaces
elec_gen_and_consump_transformed['DataSeries'] = (
    elec_gen_and_consump_transformed['DataSeries'].str.strip()
)

In [278]:
# Check leading or trailing whitespaces for the column "DataSeries"
elec_gen_and_consump_transformed['DataSeries'].unique()

array(['Electricity Generation', 'Electricity Consumption',
       'Industrial-Related', 'Manufacturing', 'Construction', 'Utilities',
       'Other Industrial-Related', 'Commerce And Service-Related',
       'Wholesale And Retail Trade', 'Accommodation And Food Services',
       'Information And Communications',
       'Financial And Insurance Activities', 'Real Estate Activities',
       'Professional, Scientific & Technical, Administration & Support Activities',
       'Other Commerce And Service-Related', 'Transport-Related',
       'Households', 'Others'], dtype=object)

The above output shows there are no more extra whitespaces in the *DataSeries* column values. One last thing we will do for this dataset is standardize column names to snake_case.



In [279]:
# Standardize the column names to snake case to have consistent naming conventions
elec_gen_and_consump_transformed = elec_gen_and_consump_transformed.rename(
    columns={
        'Year': 'year',
        'Category': 'category',
        'DataSeries': 'data_series',
        'Value': 'value'
    }
)

Now, we will move on to our next dataset, Greenhouse Gas Emissions by Gas Type Annual.

In [280]:
# View the greenhouse gas emission dataset for data understanding
ggas_emission_gas_type

,DataSeries,2023,2022,2021,2020,2019,2018,2017,2016,2015,...,2009,2008,2007,2006,2005,2004,2003,2002,2001,2000
0,Total Greenhouse Gas Emissions,55.5,58.6,58.3,53.9,55.6,56.8,56.1,54.1,54.4,...,44.2,44.1,44.5,44.6,43.1,43.9,42.3,42.3,40.6,39.7
1,Carbon Dioxide (CO2),48.2,50.4,51.2,47.7,49.7,51.2,50.9,49.8,50.2,...,41.9,42.0,42.6,42.0,40.7,41.8,40.3,40.8,39.3,38.3
2,Methane (CH4),0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,...,0.1,0.2,0.2,0.2,0.1,0.1,0.2,0.1,0.1,0.1
3,Nitrous Oxide (N2O),0.5,0.6,0.6,0.6,0.6,0.5,0.4,0.3,0.3,...,0.3,0.3,0.3,0.3,0.3,0.3,0.3,0.3,0.3,0.3
4,Hydrofluorocarbons (HFCs),4.0,4.2,3.7,3.3,3.1,2.8,2.4,2.1,1.9,...,1.0,0.8,0.7,0.6,0.5,0.4,0.3,0.3,0.2,0.1
5,Perfluorocarbons (PFCs),2.1,2.5,2.1,1.6,1.6,1.7,1.7,1.6,1.6,...,0.9,0.8,0.8,1.3,1.2,1.0,1.0,0.7,0.6,0.8
6,Sulphur Hexafluoride (SF6),0.1,0.2,0.1,0.1,0.1,0.1,0.1,0.1,0.1,...,0.0,0.0,0.0,0.1,0.1,0.0,0.0,0.0,0.0,0.0
7,Nitrogen Trifluoride (NF3),0.4,0.6,0.5,0.4,0.4,0.4,0.4,0.2,0.2,...,0.1,0.0,0.0,0.3,0.2,0.2,0.2,0.1,0.0,0.1
8,% Of Total GHG Emissions - Carbon Dioxide (CO2),86.9,86.0,87.8,88.6,89.4,90.0,90.8,91.9,92.3,...,94.7,95.4,95.7,94.0,94.4,95.3,95.2,96.4,96.7,96.4
9,% Of Total GHG Emissions - Methane (CH4),0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,...,0.3,0.4,0.4,0.3,0.3,0.3,0.4,0.3,0.3,0.3


Looking at the dataset gives us some idea of how to transform this dataset. Similar to the electricity generation and consumption dataset we transformed earlier, we will transform this dataset as follows:
1. Create a separate *year* column instead of one column for each year recorded
2. Add a new *Category* column to accurately represent what the current row of data is for

In [281]:
# Transform data as discussed
year_cols = [col for col in ggas_emission_gas_type.columns if col != 'DataSeries']

greenhouse_gas_emissions_transformed = ggas_emission_gas_type.melt(
    id_vars=['DataSeries'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Value'
)

# Create Category column
greenhouse_gas_emissions_transformed['Category'] = np.select(
    [
        greenhouse_gas_emissions_transformed['DataSeries'] == 'Total Greenhouse Gas Emissions',
        greenhouse_gas_emissions_transformed['DataSeries'].str.contains('%', regex=False)
    ],
    [
        'Total',
        'Consumption Percentage'
    ],
    default='Gas Type'
)

# Convert Year to integer
greenhouse_gas_emissions_transformed['Year'] = (
    greenhouse_gas_emissions_transformed['Year']
    .astype(int)
)

# Convert Value to numeric
greenhouse_gas_emissions_transformed['Value'] = pd.to_numeric(
    greenhouse_gas_emissions_transformed['Value'],
    errors='coerce'
)

# Rename columns to snake_case
greenhouse_gas_emissions_transformed = (
    greenhouse_gas_emissions_transformed
    .rename(columns={
        'Year': 'year',
        'Category': 'category',
        'DataSeries': 'data_series',
        'Value': 'value'
    })
)

# Reorder columns
greenhouse_gas_emissions_transformed = greenhouse_gas_emissions_transformed[
    ['year', 'category', 'data_series', 'value']
]

greenhouse_gas_emissions_transformed.head()

,year,category,data_series,value
0,2023,Total,Total Greenhouse Gas Emissions,55.5
1,2023,Gas Type,Carbon Dioxide (CO2),48.2
2,2023,Gas Type,Methane (CH4),0.1
3,2023,Gas Type,Nitrous Oxide (N2O),0.5
4,2023,Gas Type,Hydrofluorocarbons (HFCs),4.0


In [282]:
greenhouse_gas_emissions_transformed

,year,category,data_series,value
0,2023,Total,Total Greenhouse Gas Emissions,55.5
1,2023,Gas Type,Carbon Dioxide (CO2),48.2
2,2023,Gas Type,Methane (CH4),0.1
3,2023,Gas Type,Nitrous Oxide (N2O),0.5
4,2023,Gas Type,Hydrofluorocarbons (HFCs),4.0
...,...,...,...,...
355,2000,Consumption Percentage,% Of Total GHG Emissions - Nitrous Oxide (N2O),0.7
356,2000,Consumption Percentage,% Of Total GHG Emissions - Hydrofluorocarbons ...,0.4
357,2000,Consumption Percentage,% Of Total GHG Emissions - Perfluorocarbons (P...,1.9
358,2000,Consumption Percentage,% Of Total GHG Emissions - Sulphur Hexafluorid...,0.1


The above output shows that our transformation looks correct. However, if we examine the *data_series*, there are basically two measurements, one for raw data value (in million tonnes of CO2 equivalent) and other is for percentage for percentage columns. Therefore, we will decide to make a different dataset for percentage-based rows under *data_series* column. 

In [283]:
# Separate percentage and non-percentage records into two separate datasets for easier analysis
greenhouse_gas_emissions_percentage_transformed = (
    greenhouse_gas_emissions_transformed[
        greenhouse_gas_emissions_transformed['category'] == 'Consumption Percentage'
    ]
    .copy()
)

# Rename value column
greenhouse_gas_emissions_percentage_transformed = (
    greenhouse_gas_emissions_percentage_transformed
    .rename(columns={'value': 'percentage'})
)

# Remove category column (it's no longer needed)
greenhouse_gas_emissions_percentage_transformed = (
    greenhouse_gas_emissions_percentage_transformed
    .drop(columns='category')
)

In [284]:
greenhouse_gas_emissions_percentage_transformed

,year,data_series,percentage
8,2023,% Of Total GHG Emissions - Carbon Dioxide (CO2),86.9
9,2023,% Of Total GHG Emissions - Methane (CH4),0.2
10,2023,% Of Total GHG Emissions - Nitrous Oxide (N2O),1.0
11,2023,% Of Total GHG Emissions - Hydrofluorocarbons ...,7.2
12,2023,% Of Total GHG Emissions - Perfluorocarbons (P...,3.8
...,...,...,...
355,2000,% Of Total GHG Emissions - Nitrous Oxide (N2O),0.7
356,2000,% Of Total GHG Emissions - Hydrofluorocarbons ...,0.4
357,2000,% Of Total GHG Emissions - Perfluorocarbons (P...,1.9
358,2000,% Of Total GHG Emissions - Sulphur Hexafluorid...,0.1


The above output shows the correct transformation done. Next, we will do a quickcheck of whether there are any missing or duplicate records in the transformed datasets.

In [285]:
# Quick check for missing values and duplicates in the transformed datasets
datasets = {
    "Greenhouse Gas Emissions": greenhouse_gas_emissions_transformed,
    "Greenhouse Gas Emissions Percentage": greenhouse_gas_emissions_percentage_transformed
}

for name, df in datasets.items():
    print("Number of missing values in {}:".format(name))
    print(df.isna().sum())
    print("")
    print("Number of duplicate values in {}:".format(name))
    print(df.isnull().sum())
    print("")

Number of missing values in Greenhouse Gas Emissions:
year           0
category       0
data_series    0
value          0
dtype: int64

Number of duplicate values in Greenhouse Gas Emissions:
year           0
category       0
data_series    0
value          0
dtype: int64

Number of missing values in Greenhouse Gas Emissions Percentage:
year           0
data_series    0
percentage     0
dtype: int64

Number of duplicate values in Greenhouse Gas Emissions Percentage:
year           0
data_series    0
percentage     0
dtype: int64



Since there are none, we don't need to take any action. Finally, we will truncate text columns in case there are any extra whitespaces.

In [286]:
# Truncate text columns
for df in [greenhouse_gas_emissions_transformed, greenhouse_gas_emissions_percentage_transformed]:
    obj_cols = df.select_dtypes(include='object').columns
    df[obj_cols] = df[obj_cols].apply(lambda col: col.str.strip())

In [287]:
greenhouse_gas_emissions_percentage_transformed['data_series'].unique()

array(['% Of Total GHG Emissions - Carbon Dioxide (CO2)',
       '% Of Total GHG Emissions - Methane (CH4)',
       '% Of Total GHG Emissions - Nitrous Oxide (N2O)',
       '% Of Total GHG Emissions - Hydrofluorocarbons (HFCs)',
       '% Of Total GHG Emissions - Perfluorocarbons (PFCs)',
       '% Of Total GHG Emissions - Sulphur Hexafluoride (SF6)',
       '% Of Total GHG Emissions - Nitrogen Trifluoride (NF3)'],
      dtype=object)

Finally, we will remove percentage-based rows in the original transformed dataset, and ensure data types in both datasets are correct.

In [288]:
# Remove percentage records from the greenhouse and check data types for both datasets
greenhouse_gas_emissions_transformed = (
    greenhouse_gas_emissions_transformed[
        ~greenhouse_gas_emissions_transformed['data_series'].str.contains('%', regex=False)
    ]
)

print(greenhouse_gas_emissions_transformed.dtypes)
print("")
print(greenhouse_gas_emissions_percentage_transformed.dtypes)

# Rename for better readability
ggas_emissions_transformed = greenhouse_gas_emissions_transformed
ggas_emissions_percentage_transformed = greenhouse_gas_emissions_percentage_transformed

year             int32
category        object
data_series     object
value          float64
dtype: object

year             int32
data_series     object
percentage     float64
dtype: object


Now everything looks correct. We will move on to transforming our next dataset, greenhouse gas emissions, but this time by sectors.

In [289]:
# View the greenhouse gas emission by sector dataset for data understanding
ggas_emission_sector

,DataSeries,2023,2022,2021,2020
0,Primary Emissions - Power,38.7,36.5,39.2,39.8
1,Secondary Emissions - Industry,17.0,16.1,16.6,16.2
2,Secondary Emissions - Transport,2.2,2.0,2.2,2.3
3,Secondary Emissions - Buildings,12.4,11.7,12.6,13.1
4,Secondary Emissions - Household,6.1,5.8,6.6,7.1
5,Secondary Emissions - Waste And Water,0.7,0.7,0.8,0.8
6,Secondary Emissions - Others,0.2,0.3,0.4,0.3
7,Primary Emissions - Industry,47.3,49.0,44.4,44.4
8,Primary Emissions - Transport,11.7,12.0,14.2,13.7
9,Primary Emissions - Buildings,0.8,0.9,0.9,0.8


If we look at the data, for *DataSeries* column, there are primary emissions and secondary emissions. Upon checking the source https://tablebuilder.singstat.gov.sg/table/TS/M891491 , the primary and secondary emissions are defined as below:
1. Primary Emissions: Direct emissions of greenhouse gases from the sector's owned or controlled sources.
2. Secondary Emissions: The sector's share of Power sector's emissions based on its share of electricity consumption.

In simpler terms, primary emissions are greenhouse gas emissions directly produced by a sector, while secondary emissions are emissions from electricity generation that are allocated to sectors based on their electricity consumption.

As such, we will transform the dataset as follows:
1. Make a separate *year* column
2. Split the *DataSeries* column into two, one representing the type of emission (either **Primary** or **Secondary**) and other showing which sector it belongs to.
3. Create a *value* column that shows respective emission percentage.

In [290]:
# Transform Greenhouse Gas Emissions by Sector dataset
year_cols = [col for col in ggas_emission_sector.columns if col != 'DataSeries']

greenhouse_gas_emissions_by_sector_transformed = (
    ggas_emission_sector.melt(
        id_vars='DataSeries',
        value_vars=year_cols,
        var_name='year',
        value_name='value'
    )
)

# Split DataSeries into emission type and sector
greenhouse_gas_emissions_by_sector_transformed[
    ['emission_type', 'sector']
] = (
    greenhouse_gas_emissions_by_sector_transformed['DataSeries']
    .str.split(' - ', n=1, expand=True)
)

# Remove the word "Emissions" from emission_type
greenhouse_gas_emissions_by_sector_transformed['emission_type'] = (
    greenhouse_gas_emissions_by_sector_transformed['emission_type']
    .str.replace(' Emissions', '', regex=False)
)

# Convert data types
greenhouse_gas_emissions_by_sector_transformed['year'] = (
    greenhouse_gas_emissions_by_sector_transformed['year']
    .astype(int)
)

greenhouse_gas_emissions_by_sector_transformed['value'] = pd.to_numeric(
    greenhouse_gas_emissions_by_sector_transformed['value'],
    errors='coerce'
)

# Keep only the required columns
greenhouse_gas_emissions_by_sector_transformed = (
    greenhouse_gas_emissions_by_sector_transformed[
        ['year', 'emission_type', 'sector', 'value']
    ]
)

greenhouse_gas_emissions_by_sector_transformed.head()

,year,emission_type,sector,value
0,2023,Primary,Power,38.7
1,2023,Secondary,Industry,17.0
2,2023,Secondary,Transport,2.2
3,2023,Secondary,Buildings,12.4
4,2023,Secondary,Household,6.1


The output above shows that the transformation is done correctly. As usual, we will check data types, missing and duplicate values in the dataset, and truncate text columns to ensure there are no leading or trailing whitespaces.

In [291]:
# Check missing values
print("Missing values:\n", greenhouse_gas_emissions_by_sector_transformed.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:", greenhouse_gas_emissions_by_sector_transformed.duplicated().sum())

# Check data types
print("\nData types:\n", greenhouse_gas_emissions_by_sector_transformed.dtypes)

# Truncate leading/trailing whitespace in text/object columns
obj_cols = greenhouse_gas_emissions_by_sector_transformed.select_dtypes(include='object').columns
greenhouse_gas_emissions_by_sector_transformed[obj_cols] = greenhouse_gas_emissions_by_sector_transformed[obj_cols].apply(lambda col: col.str.strip())

Missing values:
 year             0
emission_type    0
sector           0
value            0
dtype: int64

Duplicate rows: 0

Data types:
 year               int32
emission_type     object
sector            object
value            float64
dtype: object


From the results, we see that nothing is wrong and needs no further action from us. Finally, we will do a simple rename of the dataset to shorten it for better readability.

In [292]:
# Shorten the dataset name for easier reference in the notebook
ggas_emissions_sector_transformed = greenhouse_gas_emissions_by_sector_transformed

Now, we will move on to our fifth dataset, per capita GDP in chained 2015 Dollars annual. As always, we will look at the data first for initial data understanding.

In [293]:
# View the Per Capita GDP in Chained 2015 Dollars Annual dataset for data understanding
per_capita_gdp_chained.head()

,DataSeries,2025,2024,2023,2022,2021,2020,2019,2018,2017,...,1969,1968,1967,1966,1965,1964,1963,1962,1961,1960
0,Per Capita GDP In Chained (2015) Dollars,97177.0,93666.0,90707.0,93861.0,93277.0,81230.0,84010.0,84004.0,81792.0,...,8575.0,7647.0,6853.0,6227.0,5794.0,5505.0,5829.0,5433.0,5193.0,4966
1,Per Capita GDP In Chained (2015) Dollars (US D...,70684.0,68131.0,65978.0,68272.0,67848.0,59085.0,61107.0,61103.0,59494.0,...,6237.0,5562.0,4985.0,4529.0,4214.0,4004.0,4240.0,3952.0,3777.0,3612
2,Per Capita Manufacturing In Chained (2015) Dol...,20462.0,19054.0,18731.0,20532.0,20664.0,17493.0,16217.0,16646.0,15626.0,...,1478.0,1223.0,1043.0,928.0,834.0,764.0,761.0,669.0,631.0,619
3,Per Capita Manufacturing In Chained (2015) Dol...,14884.0,13859.0,13625.0,14935.0,15031.0,12724.0,11796.0,12108.0,11366.0,...,1075.0,890.0,759.0,675.0,607.0,556.0,554.0,487.0,459.0,450
4,Year On Year Growth Rate Of Per Capita GDP In ...,3.7,3.3,-3.4,0.6,14.8,-3.3,0.0,2.7,4.2,...,12.1,11.6,10.1,7.5,5.2,-5.6,7.3,4.6,4.6,na


In [294]:
# Transform the dataset

# Remove US Dollar rows
per_capita_gdp_chained_clean = per_capita_gdp_chained[
    ~per_capita_gdp_chained['DataSeries'].str.contains('US Dollar', regex=False)
].copy()

# Split into value and growth rate datasets
value_df = per_capita_gdp_chained_clean[
    ~per_capita_gdp_chained_clean['DataSeries'].str.contains('Year On Year Growth Rate', regex=False)
].copy()

growth_df = per_capita_gdp_chained_clean[
    per_capita_gdp_chained_clean['DataSeries'].str.contains('Year On Year Growth Rate', regex=False)
].copy()

year_cols = [col for col in per_capita_gdp_chained.columns if col != 'DataSeries']

# ---------------------------
# Per Capita GDP/Manufacturing Values
# ---------------------------
per_capita_gdp_chained_transformed = value_df.melt(
    id_vars='DataSeries',
    value_vars=year_cols,
    var_name='year',
    value_name='value'
)

per_capita_gdp_chained_transformed['metric'] = np.where(
    per_capita_gdp_chained_transformed['DataSeries'].str.contains('Manufacturing', regex=False),
    'Manufacturing',
    'GDP'
)

per_capita_gdp_chained_transformed['year'] = per_capita_gdp_chained_transformed['year'].astype(int)
per_capita_gdp_chained_transformed['value'] = pd.to_numeric(
    per_capita_gdp_chained_transformed['value'],
    errors='coerce'
)

per_capita_gdp_chained_transformed = per_capita_gdp_chained_transformed[
    ['year', 'metric', 'value']
]

# ---------------------------
# Year-on-Year Growth Rate
# ---------------------------
per_capita_gdp_yoy_growth_rate_transformed = growth_df.melt(
    id_vars='DataSeries',
    value_vars=year_cols,
    var_name='year',
    value_name='growth_rate'
)

per_capita_gdp_yoy_growth_rate_transformed['metric'] = np.where(
    per_capita_gdp_yoy_growth_rate_transformed['DataSeries'].str.contains('Manufacturing', regex=False),
    'Manufacturing',
    'GDP'
)

per_capita_gdp_yoy_growth_rate_transformed['year'] = per_capita_gdp_yoy_growth_rate_transformed['year'].astype(int)
per_capita_gdp_yoy_growth_rate_transformed['growth_rate'] = pd.to_numeric(
    per_capita_gdp_yoy_growth_rate_transformed['growth_rate'],
    errors='coerce'
)

per_capita_gdp_yoy_growth_rate_transformed = per_capita_gdp_yoy_growth_rate_transformed[
    ['year', 'metric', 'growth_rate']
]

# Display
per_capita_gdp_chained_transformed.head(), per_capita_gdp_yoy_growth_rate_transformed.head()

(   year         metric    value
 0  2025            GDP  97177.0
 1  2025  Manufacturing  20462.0
 2  2024            GDP  93666.0
 3  2024  Manufacturing  19054.0
 4  2023            GDP  90707.0,
    year         metric  growth_rate
 0  2025            GDP          3.7
 1  2025  Manufacturing          7.4
 2  2024            GDP          3.3
 3  2024  Manufacturing          1.7
 4  2023            GDP         -3.4)

The output shows the correct transformations done. Next, just like previous datasets, we will check data types, missing and duplicate values, and truncate any text columns.

In [295]:
# per_capita_gdp_chained_transformed
print("=== per_capita_gdp_chained_transformed ===")
print("Missing values:\n", per_capita_gdp_chained_transformed.isnull().sum())
print("\nDuplicate rows:", per_capita_gdp_chained_transformed.duplicated().sum())
print("\nData types:\n", per_capita_gdp_chained_transformed.dtypes)
print("\n")

obj_cols = per_capita_gdp_chained_transformed.select_dtypes(include='object').columns
per_capita_gdp_chained_transformed[obj_cols] = per_capita_gdp_chained_transformed[obj_cols].apply(lambda col: col.str.strip())

# per_capita_gdp_yoy_growth_rate_transformed
print("=== per_capita_gdp_yoy_growth_rate_transformed ===")
print("Missing values:\n", per_capita_gdp_yoy_growth_rate_transformed.isnull().sum())
print("\nDuplicate rows:", per_capita_gdp_yoy_growth_rate_transformed.duplicated().sum())
print("\nData types:\n", per_capita_gdp_yoy_growth_rate_transformed.dtypes)
print("\n")

obj_cols = per_capita_gdp_yoy_growth_rate_transformed.select_dtypes(include='object').columns
per_capita_gdp_yoy_growth_rate_transformed[obj_cols] = per_capita_gdp_yoy_growth_rate_transformed[obj_cols].apply(lambda col: col.str.strip())

=== per_capita_gdp_chained_transformed ===
Missing values:
 year      0
metric    0
value     0
dtype: int64

Duplicate rows: 0

Data types:
 year        int32
metric     object
value     float64
dtype: object


=== per_capita_gdp_yoy_growth_rate_transformed ===
Missing values:
 year           0
metric         0
growth_rate    2
dtype: int64

Duplicate rows: 0

Data types:
 year             int32
metric          object
growth_rate    float64
dtype: object




From the output, we can see that Year on Year dataset has 2 missing values (2 records with missing values under *growth_rate* column). Therefore, we will remove those records.

In [296]:
# Remove missing values in the column "growth_rate" since it is not possible to impute the missing values with a reasonable assumption.
per_capita_gdp_yoy_growth_rate_transformed = per_capita_gdp_yoy_growth_rate_transformed.dropna(subset=['growth_rate'])

In [297]:
# Do a sanity check to ensure that the number of missing records in the transformed datasets is as expected (0)
per_capita_gdp_yoy_growth_rate_transformed.isna().sum()

year           0
metric         0
growth_rate    0
dtype: int64

Now, we will move on to the last dataset we will use, Renewable Energy Share in Total Final Energy Consumption Annual.

In [298]:
# View the Renewable Energy Share in Total Final Energy Consumption Annual dataset for data understanding
renewable_energy_share

,DataSeries,2023,2022,2021,2020,2019,2018,2017,2016,2015,2014,2013,2012,2011,2010,2009
0,Renewable Energy Share In Total Final Energy C...,1.27,1.3,0.87,0.85,0.77,0.81,0.8,0.81,0.76,0.77,0.99,0.8,0.77,0.75,0.78


After looking at the data, we already have a good idea of what we should do - unpivot the dataset (make columns into rows - year column and value column), so we will do that.

In [299]:
# Transform Renewable Energy Share in Total Final Energy Consumption dataset
year_cols = [col for col in renewable_energy_share.columns if col != 'DataSeries']

renewable_energy_share_transformed = renewable_energy_share.melt(
    id_vars='DataSeries',
    value_vars=year_cols,
    var_name='year',
    value_name='value'
)

renewable_energy_share_transformed['year'] = (
    renewable_energy_share_transformed['year']
    .astype(int)
)

renewable_energy_share_transformed['value'] = pd.to_numeric(
    renewable_energy_share_transformed['value'],
    errors='coerce'
)

renewable_energy_share_transformed = renewable_energy_share_transformed[
    ['year', 'value']
]

renewable_energy_share_transformed.head()

,year,value
0,2023,1.27
1,2022,1.30
2,2021,0.87
3,2020,0.85
4,2019,0.77


As we can see, the transformation is done. To clarify the data, e.g. for 2023, value of 1.27 means 1.27% of Singapore's total final energy consumption comes from renewable energy sources. As usual, we will check data types, missing and duplicate values. There is no need for truncation as the dataset does not have any text column.

In [300]:
# Check missing and duplicate values, and data types
print("Missing values:\n", renewable_energy_share_transformed.isnull().sum())
print("\nDuplicate rows:", renewable_energy_share_transformed.duplicated().sum())
print("\nData types:\n", renewable_energy_share_transformed.dtypes)

Missing values:
 year     0
value    0
dtype: int64

Duplicate rows: 0

Data types:
 year       int32
value    float64
dtype: object


We see that there are no issue. Below are all the datasets that we have transformed from the original raw datasets:
1. *fuel_mix_elec_gen*
2. *elec_gen_and_consump_transformed*
3. *ggas_emissions_transformed*
4. *ggas_emissions_percentage_transformed*
5. *ggas_emissions_sector_transformed*
6. *per_capita_gdp_chained_transformed*
7. *per_capita_gdp_yoy_growth_rate_transformed*
8. *renewable_energy_share_transformed*

Before exporting out the cleaned and transformed datasets, we will do a final data quality check in the next cell, checking data shape (number of rows and columns), data types, missing and duplicat values, as well as text/categorical columns to catch any truncation errors.

In [301]:
# Do a final data quality check for all transformed datasets
datasets = {
    'Annual Fuel Mix for Electricity Generation by Energy Products 2005 to Jun 2021 (Transformed)': fuel_mix_elec_gen,
    'Electricity Generation and Consumption Annual (Transformed)': elec_gen_and_consump_transformed,
    'Greenhouse Gas Emissions by Gas Type Annual (Transformed)': ggas_emissions_transformed,
    'Greenhouse Gas Emissions by Gas Type Percentage Annual (Transformed)': ggas_emissions_percentage_transformed,
    'Greenhouse Gas Emissions by Sector Annual (Transformed)': ggas_emissions_sector_transformed,
    'Per Capita GDP in Chained 2015 Dollars Annual (Transformed)': per_capita_gdp_chained_transformed,
    'Per Capita GDP in Chained 2015 Dollars YoY Growth Rate Annual (Transformed)': per_capita_gdp_yoy_growth_rate_transformed,
    'Renewable Energy Share in Total Final Energy Consumption Annual (Transformed)': renewable_energy_share_transformed,
}

for name, df in datasets.items():
    print(f"===== {name} =====")
    print("Shape:", df.shape)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nData types:")
    print(df.dtypes)

    # Check unique values for categorical columns
    object_cols = df.select_dtypes(include='object').columns

    if len(object_cols) > 0:
        print("\nUnique values in categorical columns:")
        for col in object_cols:
            print(f"\n{col}:")
            print(df[col].unique())

    print("\n" + "=" * 100 + "\n")

===== Annual Fuel Mix for Electricity Generation by Energy Products 2005 to Jun 2021 (Transformed) =====
Shape: (68, 3)

Missing values:
year               0
energy_products    0
percentage         0
dtype: int64

Duplicate rows: 0

Data types:
year                 int64
energy_products     object
percentage         float64
dtype: object

Unique values in categorical columns:

energy_products:
['Petroleum Products' 'Natural Gas' 'Coal' 'Others']


===== Electricity Generation and Consumption Annual (Transformed) =====
Shape: (391, 4)

Missing values:
year           0
category       0
data_series    0
value          0
dtype: int64

Duplicate rows: 0

Data types:
year             int32
category        object
data_series     object
value          float64
dtype: object

Unique values in categorical columns:

category:
['Total' 'Consumption Sector']

data_series:
['Electricity Generation' 'Electricity Consumption' 'Industrial-Related'
 'Manufacturing' 'Construction' 'Utilities' 'Other Indus

After examining the output, we can see that everything looks correct. However, one small step we will take is to rename row values under *data_series* column in the ggas_emissions_percentage_transformed dataset. Since the names are too long, we will get rid of the first part before " - " to shorten the values for better readability in the later analysis.

In [302]:
# Notice the long names under data_series column
ggas_emissions_percentage_transformed.head()

,year,data_series,percentage
8,2023,% Of Total GHG Emissions - Carbon Dioxide (CO2),86.9
9,2023,% Of Total GHG Emissions - Methane (CH4),0.2
10,2023,% Of Total GHG Emissions - Nitrous Oxide (N2O),1.0
11,2023,% Of Total GHG Emissions - Hydrofluorocarbons ...,7.2
12,2023,% Of Total GHG Emissions - Perfluorocarbons (P...,3.8


In [303]:
# Keep only gas types for better readability
ggas_emissions_percentage_transformed['data_series'] = (
    ggas_emissions_percentage_transformed['data_series']
    .str.split(' - ', n=1)
    .str[1]
)

Then, we will check if there are any leading or trailing whitespaces to ensure the *data_series* column is clean.

In [304]:
ggas_emissions_percentage_transformed['data_series'].unique()

array(['Carbon Dioxide (CO2)', 'Methane (CH4)', 'Nitrous Oxide (N2O)',
       'Hydrofluorocarbons (HFCs)', 'Perfluorocarbons (PFCs)',
       'Sulphur Hexafluoride (SF6)', 'Nitrogen Trifluoride (NF3)'],
      dtype=object)

Before exporting out the transformed datasets, we will do one final transformation, to renewable energy share dataset - to include a new column representing year on year growth rate, to enable more analysis. We will name this column as *yoy_growth_rate*.

In [305]:
# Ensure data is sorted by year before computing year-over-year growth
renewable_energy_share_transformed = renewable_energy_share_transformed.sort_values('year').reset_index(drop=True)

# Calculate year-over-year growth rate (% change from previous year's value)
renewable_energy_share_transformed['yoy_growth_rate'] = (renewable_energy_share_transformed['value'].pct_change() * 100).round(2)

renewable_energy_share_transformed

,year,value,yoy_growth_rate
0,2009,0.78,NaN
1,2010,0.75,-3.85
2,2011,0.77,2.67
3,2012,0.80,3.90
4,2013,0.99,23.75
5,2014,0.77,-22.22
6,2015,0.76,-1.30
7,2016,0.81,6.58
8,2017,0.80,-1.23
9,2018,0.81,1.25


One thing to note is that the **NaN** (missing value) under *yoy_growth_rate* for the first year (2009) is expected, as there is no previous year's value to calculate the year-over-year growth rate.


Finally, we will export our cleaned and transformed datasets to csv format for later tasks.

In [306]:
# # Export all cleaned and transformed datasets to CSV files for further analysis and visualization in the next steps of the project.
import os

output_dir = r"C:\Users\Admin\Desktop\net-zero emission by 2025 project\net-zero-emission-data-analytics-project\cleaned datasets"
os.makedirs(output_dir, exist_ok=True)

datasets = {
    'fuel_mix_elec_gen': fuel_mix_elec_gen,
    'elec_gen_and_consump_transformed': elec_gen_and_consump_transformed,
    'ggas_emissions_transformed': ggas_emissions_transformed,
    'ggas_emissions_percentage_transformed': ggas_emissions_percentage_transformed,
    'ggas_emissions_sector_transformed': ggas_emissions_sector_transformed,
    'per_capita_gdp_chained_transformed': per_capita_gdp_chained_transformed,
    'per_capita_gdp_yoy_growth_rate_transformed': per_capita_gdp_yoy_growth_rate_transformed,
    'renewable_energy_share_transformed': renewable_energy_share_transformed,
}

for name, df in datasets.items():
    file_path = os.path.join(output_dir, f"{name}.csv")
    df.to_csv(file_path, index=False)
    print(f"Saved: {file_path}")

Saved: C:\Users\Admin\Desktop\net-zero emission by 2025 project\net-zero-emission-data-analytics-project\cleaned datasets\fuel_mix_elec_gen.csv
Saved: C:\Users\Admin\Desktop\net-zero emission by 2025 project\net-zero-emission-data-analytics-project\cleaned datasets\elec_gen_and_consump_transformed.csv
Saved: C:\Users\Admin\Desktop\net-zero emission by 2025 project\net-zero-emission-data-analytics-project\cleaned datasets\ggas_emissions_transformed.csv
Saved: C:\Users\Admin\Desktop\net-zero emission by 2025 project\net-zero-emission-data-analytics-project\cleaned datasets\ggas_emissions_percentage_transformed.csv
Saved: C:\Users\Admin\Desktop\net-zero emission by 2025 project\net-zero-emission-data-analytics-project\cleaned datasets\ggas_emissions_sector_transformed.csv
Saved: C:\Users\Admin\Desktop\net-zero emission by 2025 project\net-zero-emission-data-analytics-project\cleaned datasets\per_capita_gdp_chained_transformed.csv
Saved: C:\Users\Admin\Desktop\net-zero emission by 2025 pro